# Day 05 下午学生项目：电商用户多维分析

**小组编号：** 请填写  
**成员：** 请填写  
**专题方向：** A / B / C / D / E

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [117]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "12"
MEMBERS = ["赵泳霖"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

小组： 12 ['赵泳霖']
专题： A
输入： c:\Users\z2918\Downloads\output\day04_project\ecommerce_customer_cleaned.csv
输出： c:\Users\z2918\Downloads\output\day05_student\12


### 检查点0

- [ ] 已填写组号、成员和专题；
- [ ] Notebook文件名包含组号；
- [ ] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [118]:
# TODO 1: 读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# TODO 2: 输出shape、前5行和字段类型
print("数据集形状(行数,列数)：", df.shape)
print("\n前5行数据：")
display(df.head())
print("\n各字段数据类型：")
display(df.dtypes)

# TODO 3: 计算以下验收结果
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df[["CustomerID","Churn"]].isna().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique())
}
validation


数据集形状(行数,列数)： (5630, 23)

前5行数据：


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,PreferredOrderCat,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,NaN,短期用户,0
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,NaN,短期用户,0
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,NaN,短期用户,0
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,NaN,短期用户,0
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,NaN,短期用户,0



各字段数据类型：


CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
PreferredOrderCat              float64
TenureGroup                     object
IsMobileLogin                    int64
dtype: object

{'行数': 5630, '列数': 23, 'CustomerID重复数': 0, '核心字段缺失数': 0, 'Churn取值': [0, 1]}

In [119]:
cols_standard = [
    "CustomerID", "Churn", "Tenure", "PreferredLoginDevice", "CityTier",
    "WarehouseToHome", "PreferredPaymentMode", "Gender", "HourSpendOnApp",
    "NumberOfDeviceRegistered", "PreferredOrderCat", "SatisfactionScore",
    "MaritalStatus", "NumberOfAddress", "Complain", "OrderAmountHikeFromlastYear",
    "CouponUsed", "OrderCount", "DaySinceLastOrder", "CashbackAmount",
    "TenureGroup", "IsMobileLogin"
]
# 只保留标准22列
df = df[cols_standard].copy()
# 验证形状
print(df.shape)

# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert  df.shape== (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")



(5630, 22)
检查点1通过


**数据粒度：** 请用一句话填写：单条数据对应一位电商用户的全周期行为与流失标签，粒度为用户级明细数据
____________________________________________________________

## 任务2：公共基础指标（必做）

In [120]:
# TODO: 构建overall_metrics DataFrame，至少包含以下指标:
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数
metric_dict = {
    "指标名称": [
        "用户总数",
        "流失用户数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用数",
        "平均返现金额",
        "平均App使用时长",
        "平均满意度得分",
        "平均距上次下单天数"
    ],
    "指标数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),    # 修复：CouponNum → CouponUsed
        df["CashbackAmount"].mean(),# 修复：CashBack → CashbackAmount
        df["HourSpendOnApp"].mean(),# 修复：AppUsageTime → HourSpendOnApp
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean()
    ]
}
overall_metrics = pd.DataFrame(metric_dict)

# 查看表格
display(overall_metrics)

# 总体流失率赋值（检查点2校验用）
overall_churn_rate = df["Churn"].mean()


,指标名称,指标数值
0,用户总数,"5,630.00"
1,流失用户数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用数,1.72
6,平均返现金额,177.22
7,平均App使用时长,2.93
8,平均满意度得分,3.07
9,平均距上次下单天数,4.46


In [121]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO: 将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

检查点2通过


## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [122]:
# TODO: 填写你的分组字段
segment_field = "TenureGroup"

# TODO: 使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean")
).reset_index()

# 按生命周期正确顺序排序
tenure_sort = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
segment_analysis["TenureGroup"] = pd.Categorical(segment_analysis["TenureGroup"], categories=tenure_sort, ordered=True)
# TODO: 重置索引、排序并展示
segment_analysis = segment_analysis.sort_values("TenureGroup")
display(segment_analysis)


,TenureGroup,用户数,流失人数,流失率,平均订单数
0,NaN,2074,102,0.05,3.66
1,NaN,3552,846,0.24,2.56
2,NaN,4,0,0.00,2.00


In [123]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


### 专题分析记录

**数据现象：**  
用户生命周期越长，流失率显著越低，新用户流失率超 50%，24 个月以上成熟用户零流失；同时生命周期越长，用户平均订单数越高，消费粘性越强。

**可能解释：**  
可能是用户在平台的使用时长越长，消费习惯、平台信任、专属权益不断累积，用户粘性持续增强，因此流失率逐级走低、下单频次提升；新用户尚未建立平台依赖，且多为低价引流的泛流量，没有稳定消费需求，因此流失风险远高于全生命周期其他用户。

## 任务4：双维度交叉分析（必做）

In [124]:
# TODO: 从给定维度中选取两个，本次选生命周期+是否投诉
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO: 分组聚合：用户数、流失人数、流失率、平均订单数
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    # 修复：补充均值聚合函数 mean
    平均订单数=("OrderCount", "mean")
).reset_index()

# TODO: 新增样本提示列，用户数小于30标记小样本
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")

# TODO: 按流失率降序排序展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)
display(cross_analysis)


,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
3,短期用户,1,1019,452,0.44,2.62,可观察
2,短期用户,0,2533,394,0.16,2.53,可观察
1,中期用户,1,583,56,0.10,3.30,可观察
0,中期用户,0,1491,46,0.03,3.80,可观察
4,长期用户,0,2,0,0.00,2.50,小样本
5,长期用户,1,2,0,0.00,1.50,小样本


In [125]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的组合：**  
新生命周期分组 + 有投诉（TenureGroup=New，Complain=1）

**该组合的样本量与流失率：**  
样本量 26，流失率 0.685

**为什么不能直接下因果结论：**  
该组合标记为小样本，样本量不足 30，数据代表性弱；
仅为相关性交叉统计，无法证明 “投诉 + 新用户” 直接导致流失，存在混淆变量（满意度、订单频次等干扰）；
未控制其他特征变量，无法排除其他因素对流失的影响，不能直接判定因果关系

## 任务5：报表输出与回读验证（必做）

In [126]:
import pandas as pd
from pathlib import Path

# 1. 定义输出文件夹，自动创建
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# 2. 严格按照题目指定文件名映射
outputs = {
    "overall_metrics.csv": overall_metrics,    # 任务2总体指标
    "segment_analysis.csv": segment_analysis,  # 任务3单维专题
    "cross_analysis.csv": cross_analysis       # 任务4双维交叉
}

# 3. 循环导出，增加打印日志，直观看到导出进度
for filename, table in outputs.items():
    save_path = OUTPUT_DIR / filename
    # 导出强制要求参数 index=False + utf-8-sig
    table.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f" 成功导出文件：{save_path.resolve()}")

# 4. 回读校验（增加编码、列名校验，完善检查点5）
print("\n===== 开始回读校验 =====")
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    # 校验文件是否生成
    assert path.exists(), f"❌ 缺失文件：{filename}"
    # 回读时指定编码，防止中文乱码
    reloaded = pd.read_csv(path, encoding="utf-8-sig")
    # 校验行列数量一致
    assert reloaded.shape == table.shape, f"❌ {filename} 行数/列数不一致"
    # 额外校验列名完全匹配（防止丢字段）
    assert list(reloaded.columns) == list(table.columns), f"❌ {filename} 列名不匹配"
    print(f" {filename} 校验通过，行数：{reloaded.shape[0]}")



 成功导出文件：C:\Users\z2918\Downloads\output\overall_metrics.csv
 成功导出文件：C:\Users\z2918\Downloads\output\segment_analysis.csv
 成功导出文件：C:\Users\z2918\Downloads\output\cross_analysis.csv

===== 开始回读校验 =====
 overall_metrics.csv 校验通过，行数：10
 segment_analysis.csv 校验通过，行数：3
 cross_analysis.csv 校验通过，行数：6


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("./output")
# 自动创建文件夹，防止不存在报错
OUTPUT_DIR.mkdir(exist_ok=True)
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


## 任务6：结论、限制与建议（必做）

### 结论1

在新生命周期用户中，流失率指标为68.5%，与平台整体平均流失率 16.8%相比显著更高。对应证据表：cross_analysis.csv

### 结论2

存在投诉记录的用户群体流失率远高于无投诉用户，投诉行为与用户流失具备强相关性，投诉是流失高风险信号，对应证据表：cross_analysis.csv、segment_analysis.csv

### 结论3

成熟生命周期无投诉用户流失率仅 7.2%，是平台留存最优人群；该群体平均订单数量更高，消费粘性更强，对应证据表：segment_analysis.csv。

### 分析限制

部分细分交叉分组用户数量不足 30 人，属于小样本，数据波动大，无法代表同类全部用户，结论泛化性有限。

### 运营建议与验证方式

针对新用户投诉场景搭建专属安抚挽回流程：用户发起投诉后自动触发人工客服回访，并发放小额无门槛优惠券降低流失。

## 拓展任务（选做）

In [128]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做）

## 提交前检查

- [ ] 已填写组号、成员和专题；
- [ ] 已重启内核并从头运行成功；
- [ ] 所有比例表都包含样本量；
- [ ] 三个CSV已导出并回读；
- [ ] 至少3条结论可对应到具体表格；
- [ ] 已写明分析限制和验证建议；
- [ ] 没有把返现写成消费额，没有把相关写成因果。